In [5]:
from datasets import load_dataset
from pathlib import Path
from tqdm.auto import tqdm
import re
import unicodedata
import random


In [ ]:


def clean_text(text):

    """ Here i clean the data, remove punctuations, aphostrophes, commas, empty spaces etc. 
        Then i split the data into 80% train, 10% validation for finding the right weights in the
        interpolation function in the n-gram model and finally 10% for testing. 

        I validate on :top-K word suggestions and saved keywords
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"')
    text = text.replace("”", '"')

    text = text.replace("'", "")

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def save_lines(lines, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")


def main():
    output_dir = Path("data/splits")
    output_dir.mkdir(parents=True, exist_ok=True)

    """ I STARTED SMALL WILL DO MORE LATER"""
    
    dataset = load_dataset("roneneldan/TinyStories", split="train[:20000]")

    cleaned_stories = []

    for row in tqdm(dataset, desc="Cleaning TinyStories"):
        text = row["text"].strip()

        if not text:
            continue

        cleaned = clean_text(text.replace("\n", " "))

        if cleaned:
            cleaned_stories.append(cleaned)

    random.seed(42)
    random.shuffle(cleaned_stories)

    n = len(cleaned_stories)

    train_end = int(0.8 * n)
    val_end = int(0.9 * n)

    train_data = cleaned_stories[:train_end]
    val_data = cleaned_stories[train_end:val_end]
    test_data = cleaned_stories[val_end:]

    save_lines(train_data, output_dir / "tinystories_train.txt")
    save_lines(val_data, output_dir / "tinystories_val.txt")
    save_lines(test_data, output_dir / "tinystories_test.txt")

    print("Saved splits:")
    print("Train:", len(train_data))
    print("Val:  ", len(val_data))
    print("Test: ", len(test_data))


if __name__ == "__main__":
    main()

Cleaning TinyStories:   0%|          | 0/20000 [00:00<?, ?it/s]

Saved splits:
Train: 16000
Val:   2000
Test:  2000
